# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal checks (see code below):**
1. Staleness (`content_age_days`) vs `is_declining` — Verdict: **OPPOSITE**. Younger content
   (<90 days) declines more often (21.3%, n=73,955) than older content (13.9%, n=218,699+25,462).
   Likely survivorship: old content still standing already survived the weak ones. Dropped
   from the rule.
2. CTR-vs-position (`ctr_trailing` at good position ≤20) vs `is_declining` — Verdict:
   **CONFIRMED**. Good-position + low-CTR pages decline nearly 2x as often as good-position +
   high-CTR pages (34.3% vs 17.5%, n=15,274 vs n=18,954).

**Rule (plain words):** A page in good search position (top 20) with below-median CTR is
worth reviewing for a CTR fix — it's getting seen but not clicked.

**Reason codes the rule can output:**
- `low_ctr_good_position` — good position, weak CTR → action: review_ctr_fix
- `no_flag` — condition not met → action: monitor

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DECISION_DAY = '2026-03-15'

features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_clicks ELSE 0 END)      AS clk_trailing,
            AVG(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_avg_position END)       AS pos_trailing,
            MAX(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END)                            AS has_ga4_data
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT
        fx.*,
        DATE_DIFF('day', dc.content_created_date, DATE '{DECISION_DAY}') AS content_age_days
    FROM fx
    JOIN read_parquet('{REL}/dim_content.parquet') dc
        ON fx.content_hash_id = dc.content_hash_id
""").df()

labels = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date > DATE '{DECISION_DAY}' THEN gsc_impressions ELSE 0 END) AS imp_after
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
""").df()

merged = features.merge(labels, on=['client_hash_id', 'content_hash_id'])
merged['is_declining'] = (merged['imp_after'] < 0.8 * merged['imp_trailing']).astype(int)
merged['ctr_trailing'] = merged['clk_trailing'] / merged['imp_trailing'].replace(0, pd.NA)

print(f"Total rows: {len(merged)}")

def age_bucket(days):
    if pd.isna(days) or days < 0:
        return "unknown"
    if days < 90:
        return "1_young_<90d"
    if days < 365:
        return "2_mid_90-365d"
    return "3_old_365d+"

merged['age_tier'] = merged['content_age_days'].apply(age_bucket)

staleness_table = (
    merged.groupby('age_tier')
    .agg(n=('is_declining', 'size'), decline_rate=('is_declining', 'mean'))
    .round(3)
)
print("\n--- Signal 1: Staleness (content_age_days) vs is_declining ---")
print(staleness_table)

visible = merged[(merged['pos_trailing'] > 0) & (merged['imp_trailing'] >= 500)].copy()
ctr_median = visible['ctr_trailing'].median()

def ctr_position_bucket(row):
    good_position = row['pos_trailing'] <= 20
    low_ctr = row['ctr_trailing'] < ctr_median
    if good_position and low_ctr:
        return "good_pos_low_ctr"
    if good_position and not low_ctr:
        return "good_pos_high_ctr"
    return "other_position"

visible['ctr_pos_tier'] = visible.apply(ctr_position_bucket, axis=1)

ctr_table = (
    visible.groupby('ctr_pos_tier')
    .agg(n=('is_declining', 'size'), decline_rate=('is_declining', 'mean'))
    .round(3)
)
print(f"\n--- Signal 2: CTR-vs-position (median CTR threshold: {ctr_median:.4f}) ---")
print(f"Visible rows (imp_trailing>=500, pos_trailing>0): {len(visible)}")
print(ctr_table)

# --- Build the score, rank, write the CSV ---
import os
import json
import numpy as np

scored = merged[merged['imp_trailing'] > 0].copy()

visible_cohort = scored[(scored['imp_trailing'] >= 500) & (scored['pos_trailing'] > 0)]
ctr_median_full = visible_cohort['ctr_trailing'].median()
print(f"\nCTR median threshold (visible cohort, n={len(visible_cohort)}): {ctr_median_full:.4f}")

def make_flag(row):
    is_visible    = row['imp_trailing'] >= 500
    good_position = (row['pos_trailing'] > 0) and (row['pos_trailing'] <= 20)
    low_ctr       = pd.notna(row['ctr_trailing']) and (row['ctr_trailing'] < ctr_median_full)
    return is_visible and good_position and low_ctr

scored['flag_ctr_fix'] = scored.apply(make_flag, axis=1)
scored['score'] = scored['flag_ctr_fix'].astype(int) * scored['imp_trailing']
scored['reason_code'] = scored['flag_ctr_fix'].map({True: 'low_ctr_good_position', False: 'no_flag'})
scored['action']      = scored['flag_ctr_fix'].map({True: 'review_ctr_fix',        False: 'monitor'})

ranked = scored.sort_values('score', ascending=False).reset_index(drop=True)

def precision_at_k(labels, k):
    return np.asarray(labels)[:k].mean()

k = 50
p_at_50   = precision_at_k(ranked['is_declining'].values, k)
base_rate = ranked['is_declining'].mean()
print(f"Precision@{k}: {p_at_50:.3f}")
print(f"Base rate: {base_rate:.3f}")
print(f"Flagged rows: {int(scored['flag_ctr_fix'].sum())} / {len(scored)}")

os.makedirs('work/outputs', exist_ok=True)
output_cols = ['client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action',
               'imp_trailing', 'clk_trailing', 'pos_trailing', 'ctr_trailing', 'is_declining']
ranked[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

metrics = {
    "decision_day": DECISION_DAY,
    "rule": "imp_trailing>=500 AND good_position (<=20) AND low_ctr (< median of visible cohort) -> review_ctr_fix",
    "n_scored_rows": int(len(scored)),
    "n_flagged": int(scored['flag_ctr_fix'].sum()),
    "precision_at_50": round(float(p_at_50), 3),
    "base_rate": round(float(base_rate), 3),
    "ctr_median_threshold": round(float(ctr_median_full), 4),
}
with open('work/outputs/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

ranked.head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top10 = ranked.head(10).copy()

review_notes = [
    "High impressions (86.9k) at a strong position (5.8) but almost no clicks (CTR 0.06%) — "
    "textbook CTR-fix candidate. Would be wrong if the title/meta already changed recently and "
    "this is stale pre-change data.",

    "Zero clicks despite 83.8k impressions and decent position (8.6) — the most extreme case in "
    "the list. Would be wrong if this page targets a non-clickable intent (e.g. it already "
    "answers the query in the snippet, so users don't need to click through).",

    "Very strong position (3.5) but still flagged for low CTR (0.16%) — a top-3 ranking should "
    "pull much higher CTR. Would be wrong if the title is already optimized and the low CTR "
    "reflects a branded/navigational query where users type the URL directly instead.",

    "Excellent position (2.8) yet CTR is only 0.02%, the lowest in the top 10 — an unusually "
    "large gap. Would be wrong if this is a low-commercial-intent query (e.g. definitional) "
    "where high impressions with low clicks is simply normal query behavior, not a fixable issue.",

    "Borderline-good position (18.2, near our <=20 cutoff) with 72.9k impressions — a weaker "
    "case than the top 3. Would be wrong if position 18 is too volatile day-to-day for the "
    "CTR signal to mean anything stable.",

    "Same borderline position (18.3) pattern as the row above — flagged mainly because of scale "
    "(70.2k impressions), not because the position is clearly 'good'. Would be wrong if the "
    "position tier boundary (<=20) is too generous and this belongs in 'other_position' instead.",

    "Oldest page in the list (394 days) with 58.6k impressions but essentially one click ever "
    "(CTR 0.002%) — could be a decayed featured snippet or an outdated title. Would be wrong if "
    "the single click was a bot/crawler hit rather than a real user, making the CTR look worse "
    "than it is.",

    "Young page (67 days), strong position (9.0), meaningful impressions (55.7k) — a fairly "
    "clean case. Would be wrong if the page is still in an early ranking-fluctuation phase where "
    "CTR hasn't stabilized yet.",

    "Good position (4.0) and older content (262 days) with low CTR (0.03%) — fits the rule "
    "cleanly. Would be wrong if this query has high branded-search competition pulling clicks "
    "toward a competitor's more recognizable title instead of a content problem.",

    "Good position (4.1), young page (62 days), CTR 0.19% — the highest CTR in the top 10, so "
    "arguably the weakest case for urgency here. Would be wrong if 0.19% is actually within the "
    "normal range for this content_type and the median threshold (0.20%) is too aggressive "
    "site-wide."
]

top10['review_note'] = review_notes
top10[['content_hash_id', 'action', 'reason_code', 'review_note']]

,content_hash_id,action,reason_code,review_note
0,content_7c6373141eae744a,review_ctr_fix,low_ctr_good_position,High impressions (86.9k) at a strong position ...
1,content_9c057b66c30a3abb,review_ctr_fix,low_ctr_good_position,Zero clicks despite 83.8k impressions and dece...
2,content_acbcc847f8996314,review_ctr_fix,low_ctr_good_position,Very strong position (3.5) but still flagged f...
3,content_34a70fea29d15f24,review_ctr_fix,low_ctr_good_position,Excellent position (2.8) yet CTR is only 0.02%...
4,content_5e1c049f62e33b11,review_ctr_fix,low_ctr_good_position,"Borderline-good position (18.2, near our <=20 ..."
5,content_82e35c4845e6c391,review_ctr_fix,low_ctr_good_position,Same borderline position (18.3) pattern as the...
6,content_8e1334d6356668e3,review_ctr_fix,low_ctr_good_position,Oldest page in the list (394 days) with 58.6k ...
7,content_65c75874a23fca87,review_ctr_fix,low_ctr_good_position,"Young page (67 days), strong position (9.0), m..."
8,content_1642f339bd6e7c8d,review_ctr_fix,low_ctr_good_position,Good position (4.0) and older content (262 day...
9,content_f43118e089ecc69a,review_ctr_fix,low_ctr_good_position,"Good position (4.1), young page (62 days), CTR..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# --- Weak picks: which top-10 rows turned out wrong? ---
weak_picks = top10[top10['is_declining'] == 0][['content_hash_id', 'action', 'reason_code', 'review_note']]
print(f"Weak picks in top 10: {len(weak_picks)} / 10")
print("(is_declining == 0 despite being flagged — the rule said 'review', the outcome says 'stable')")
weak_picks

# --- Leakage check ---
# 1. Every feature used in the score comes from BEFORE or ON the decision day only.
feature_cols_used_in_score = ['imp_trailing', 'pos_trailing', 'ctr_trailing']
print("\nFeatures used in the rule/score:", feature_cols_used_in_score)
print("All three are built from rows where report_date <= DECISION_DAY only — see the")
print("CASE WHEN f.report_date <= DATE(DECISION_DAY) filters in the feature query above.")

# 2. Confirm the label and the future-window column never entered the scoring logic.
excluded_from_score = ['imp_after', 'is_declining']
score_formula_columns = {'flag_ctr_fix', 'score', 'reason_code', 'action'}
leaked = [c for c in excluded_from_score if c in score_formula_columns]
print(f"\nLabel/future columns excluded from scoring: {excluded_from_score}")
print(f"Leak check result: {'FAIL - leakage found' if leaked else 'PASS - no leakage'}")

# 3. Confirm no product-computed flags (health_score, priority_score, needs_ctr_fix, etc.)
#    were used — this dataset never ships them, so there is nothing to strip out, but we
#    state it explicitly for the record.
print("\nNo FlyRank product flags (health_score, priority_score, needs_ctr_fix, is_quick_win)")
print("exist in this warehouse release — only observable signals were available to begin with.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.